In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
from collections import Counter
from PIL import Image

BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
IMG_SIZE = 224
SEED = 42

base_search_dir = "/kaggle/input" if os.path.exists("/kaggle/input") else "./"

all_image_paths = glob.glob(os.path.join(base_search_dir, "**", "*.*"), recursive=True)
all_image_paths = [f for f in all_image_paths if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]

if len(all_image_paths) == 0:
    raise ValueError(f"CRITICAL ERROR: No images found anywhere inside {base_search_dir}!")

discovered_classes = set()
for f in all_image_paths:
    parent_folder = os.path.basename(os.path.dirname(f))
    if parent_folder.lower() not in ['training', 'testing', 'train', 'test', 'val']:
        discovered_classes.add(parent_folder)

CLASSES = sorted(list(discovered_classes))
print(f"Automatically discovered {len(CLASSES)} classes: {CLASSES}")

class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASSES)}

all_files = []
all_labels = []

for f in all_image_paths:
    parent_folder = os.path.basename(os.path.dirname(f))
    if parent_folder in class_to_idx:
        all_files.append(f)
        all_labels.append(class_to_idx[parent_folder])

print(f"Successfully loaded {len(all_files)} images.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), 
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class BrainTumorDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

class_counts = Counter(all_labels)
class_weights = []
for i in range(len(CLASSES)):
    count = class_counts.get(i, 0)
    weight = len(all_labels) / count if count > 0 else 1.0 # Protect against zero div
    class_weights.append(weight)

weights_tensor = torch.FloatTensor(class_weights).to(device)

X_train, X_val, y_train, y_val = train_test_split(all_files, all_labels, test_size=0.2, random_state=SEED, stratify=all_labels)

train_dataset = BrainTumorDataset(X_train, y_train, transform=train_transforms)
val_dataset = BrainTumorDataset(X_val, y_val, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
class BrainTumorClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super(BrainTumorClassifier, self).__init__()
        weights = models.ResNet50_Weights.DEFAULT
        self.base_model = models.resnet50(weights=weights)
        
        num_ftrs = self.base_model.fc.in_features
        self.base_model.fc = nn.Sequential(
            nn.Dropout(0.5), # Regularization
            nn.Linear(num_ftrs, num_classes)
        )
        
    def forward(self, x):
        return self.base_model(x)

model = BrainTumorClassifier(num_classes=len(CLASSES)).to(device)
print("ResNet50 Model initialized!")

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)


In [ ]:
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    # Train
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
    
    epoch_loss = running_loss / len(train_dataset)
    
    # Evaluate
    model.eval()
    val_loss = 0.0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            
            val_loss += criterion(outputs, labels).item() * inputs.size(0)
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    val_loss = val_loss / len(val_dataset)
    
    # Metrics calculation
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='macro') 
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro')
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"--> Accuracy: {acc:.4f} | F1 Score: {f1:.4f} (Precision: {precision:.4f}, Recall: {recall:.4f})")
    
    # Save the highest F1-Score model
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "/kaggle/working/best_tumor_model.pth")

print("Training cycle completed.")


In [ ]:

if os.path.exists("/kaggle/working/best_tumor_model.pth"):
    model.load_state_dict(torch.load("/kaggle/working/best_tumor_model.pth", map_location=device))
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("Confusion Matrix (Best Model)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show() 


In [ ]:
# --- Grad-CAM Cell (Random Sampling) ---
import numpy as np
import cv2

model.eval()
inputs, labels = next(iter(val_loader))

idx = np.random.randint(0, len(inputs)) 
sample_input = inputs[idx:idx+1].to(device)

cam_gen = GradCam(model, model.base_model.layer4[-1])
cam, pred = cam_gen(sample_input)


img = np.clip(np.array([0.229, 0.224, 0.225]) * inputs[idx].permute(1,2,0).numpy() + np.array([0.485, 0.456, 0.406]), 0, 1)


heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[..., ::-1] / 255.0


plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title(f"True Label: {CLASSES[labels[idx]]}")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(heatmap * 0.4 + img * 0.6) # Overlay heatmap on MRI
plt.title(f"AI Attention Map (Pred: {CLASSES[pred]})")
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# 1. Load the trained weights
model.load_state_dict(torch.load("best_tumor_model.pth", map_location=device))
model.eval()

# 2. Get predictions
all_p, all_t = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        out = model(inputs.to(device))
        all_p.extend(out.argmax(1).cpu().numpy())
        all_t.extend(labels.numpy())

# 3. Print only the requested metrics
print(f"Final Accuracy: {accuracy_score(all_t, all_p):.4f}")
print(f"Final F1-Score: {f1_score(all_t, all_p, average='macro'):.4f}")


In [8]:
import torch
import torch.nn as nn
import numpy as np
import cv2
from torchvision import models
from PIL import Image
import gradio as gr
import glob
import os

# ==========================================
# 1. MATCHED ARCHITECTURE
# ==========================================
class HybridResNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients, self.activations = None, None
        self.target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o))
        self.target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0]))

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None: class_idx = output.argmax(dim=1).item()
        self.model.zero_grad()
        output[0, class_idx].backward()
        
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations.detach().clone()
        for i in range(activations.shape[1]): activations[:, i, :, :] *= pooled_gradients[i]
        
        heatmap = torch.mean(activations, dim=1).squeeze()
        heatmap = np.maximum(heatmap.detach().cpu().numpy(), 0)
        heatmap /= np.max(heatmap) + 1e-8
        
        probs = torch.softmax(output, dim=1)
        return heatmap, class_idx, probs[0, class_idx].item(), probs[0].tolist()

# ==========================================
# 2. IMAGE UTILITIES (Fixed Precision)
# ==========================================
def preprocess_image(image, use_clahe=True):
    img = np.array(image)
    if use_clahe:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img = np.stack([clahe.apply(img[:,:,i]) for i in range(3)], axis=-1)
    
    display_img = img.copy()
    
    img = cv2.resize(img, (224, 224)).astype(np.float32) / 255.0
    # Explicitly use float32 for normalization to avoid DoubleTensor error
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img = (img - mean) / std
    
    tensor = torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0)
    return tensor.float(), display_img # <-- Final precision fix

def overlay_heatmap(image, heatmap, alpha=0.5):
    heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(image, 1 - alpha, heatmap, alpha, 0)

# ==========================================
# 3. GLOBAL SETUP 
# ==========================================
DATASET_PATH = "/kaggle/input/datasets/purvanshjoshi1/brain-tumor-model-weights"
CLASSES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = HybridResNet()
pth_files = glob.glob(os.path.join(DATASET_PATH, "*.pth"))
if pth_files:
    MODEL_PATH = pth_files[0]
    try:
        state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
        new_state_dict = {k.replace("base_model.", "backbone."): v for k, v in state_dict.items()}
        model.load_state_dict(new_state_dict, strict=True)
        print(f" AI CORE ONLINE: {os.path.basename(MODEL_PATH)} loaded perfectly.")
    except Exception as e:
        print(f" Load Error: {e}")
else:
    print(f" Error: Model weights not found.")

model.to(DEVICE).eval()
cam = GradCAM(model, model.backbone.layer4)

# ==========================================
# 4. ROBUST APP LOGIC
# ==========================================
def analyze(img, use_clahe, alpha):
    try:
        if img is None: return None, {}, "Offline"
        
        # Preprocess and Ensure Float precision
        tensor, processed = preprocess_image(img, use_clahe)
        tensor = tensor.to(DEVICE)
        
        # Inference
        heatmap, idx, conf, all_probs = cam.generate(tensor)
        
        conf_map = {CLASSES[i]: float(all_probs[i]) for i in range(4)}
        overlay = overlay_heatmap(processed, heatmap, alpha)
        
        gallery = [(img, "Original"), (processed, "Enhanced"), (overlay, f"Grad-CAM: {CLASSES[idx]}")]
        return gallery, conf_map, f"Detection: {CLASSES[idx]} ({conf:.2%})"
    except Exception as e:
        return None, {}, f" Analysis Error: {str(e)}"

# ==========================================
# 5. UI
# ==========================================
with gr.Blocks(theme=gr.themes.Soft(primary_hue="cyan")) as demo:
    gr.Markdown("#  Clinical Brain Tumor Dashboard")
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(label="Upload Scan", type="pil")
            clahe_toggle = gr.Checkbox(label="Enable CLAHE", value=True)
            alpha_slider = gr.Slider(0.1, 0.9, 0.5, label="Heatmap Alpha")
            btn = gr.Button(" EXECUTE ANALYSIS", variant="primary")
        with gr.Column():
            lbl = gr.Label(num_top_classes=4, label="Diagnosis")
            gal = gr.Gallery(columns=3, label="Visual Results")
            stat = gr.Textbox(label="System Status")
    btn.click(analyze, [input_img, clahe_toggle, alpha_slider], [gal, lbl, stat])

# NOTE: If you still see a "RuntimeError: Event loop" error, 
# simply Restart the Kaggle Kernel and run again.
demo.launch(share=True)


 AI CORE ONLINE: best_tumor_model.pth loaded perfectly.


/tmp/ipykernel_55/1854443989.py:128: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="cyan")) as demo:


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://e97d35d305c43bd55a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

In [6]:
import torch
import os

# Update this to your actual path
MODEL_PATH = "/kaggle/input/datasets/purvanshjoshi1/brain-tumor-model-weights/best_tumor_model.pth"

if os.path.exists(MODEL_PATH):
    state_dict = torch.load(MODEL_PATH, map_location='cpu')
    print("--- MODEL STRUCTURE DIAGNOSTIC ---")
    for k, v in state_dict.items():
        if "fc" in k or "classifier" in k or "head" in k:
            print(f"Key: {k:40} | Shape: {v.shape}")
else:
    print("File not found. Please check your path.")


--- MODEL STRUCTURE DIAGNOSTIC ---
Key: base_model.fc.1.weight                   | Shape: torch.Size([4, 2048])
Key: base_model.fc.1.bias                     | Shape: torch.Size([4])
